# Hotel review → amenity mention starter

This notebook is a debugging-first starter for a **2-stage pipeline**:

1. **Retrieve** likely hotel amenities for each review chunk using embeddings  
2. **Verify** each candidate with a cheap structured-output model call

It is wired to your uploaded schema:

- `Description_PROC.csv` for hotel amenity inventory
- `Reviews_PROC.csv` for review text keyed by `eg_property_id`


In [33]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

DESCRIPTION_CSV = "../sources/Description_PROC.csv"
REVIEWS_CSV = "../sources/Reviews_PROC.csv"

DESCRIPTION_CSV, REVIEWS_CSV

('../sources/Description_PROC.csv', '../sources/Reviews_PROC.csv')

In [34]:
from hotel_review_amenity_pipeline_v2 import (
    load_input_data,
    build_amenity_catalog,
    build_review_chunk_table,
    get_client,
    embed_texts,
    build_hotel_amenity_index,
    retrieve_top_amenity_candidates,
    verify_candidates_dataframe,
    process_reviews,
    build_unique_amenity_vocab,
    bootstrap_amenity_mapping,
    canonicalize_amenity_vocab_with_model,
    save_amenity_mapping,
    load_amenity_mapping,
)

client = get_client(os.getenv("api_key"))

description_df, reviews_df = load_input_data(DESCRIPTION_CSV, REVIEWS_CSV)

print(description_df.shape)
print(reviews_df.shape)
display(description_df.head(2))
display(reviews_df.head(5))

(13, 31)
(5999, 6)


,eg_property_id,guestrating_avg_expedia,city,province,country,star_rating,area_description,property_description,popular_amenities_list,property_amenity_accessibility,...,property_amenity_spa,property_amenity_things_to_do,check_in_start_time,check_in_end_time,check_out_time,check_out_policy,pet_policy,children_and_extra_bed_policy,check_in_instructions,know_before_you_go
0,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,8.4,Pompei,NaN,Italy,NaN,"located in |MASK|, hotel |MASK| |MASK| is in t...","hotel with a restaurant, steps from pompeii ar...","[""ac"", ""bar"", ""breakfast_included"", ""frontdesk...","[""this property does not have elevators""]",...,NaN,"[""shopping""]",2:00 PM,midnight,11:00 AM,"[""Check-out before 11 AM""]","[""Pets not allowed""]","[""<ul><li>Children are welcome.</li></ul><ul><...","[""front desk staff will greet guests on arriva...","[""all guests, including children, must be pres..."
1,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,9.2,Broomfield,Colorado,United States,4.0,"located in broomfield, |MASK| |MASK| is in the...","suburban resort with 3 restaurants, in the vic...","[""ac"", ""bar"", ""breakfast_available"", ""business...","[""assistive listening devices available"", ""ele...",...,"[""6 treatment rooms"", ""aromatherapy"", ""body sc...","[""2 outdoor pools"", ""24-hour fitness center"", ...",4:00 PM,anytime,11:00 AM,"[""A late check-out fee will be charged"", ""Chec...","[""1 total (up to 70 lbs per pet)"", ""Food and w...","[""<ul><li>Children are welcome.</li></ul><ul><...","[""front desk staff will greet guests on arriva...","[""a car is not required for transportation to ..."


,eg_property_id,acquisition_date,lob,rating,review_title,review_text
0,9a0043fd4258a1286db1e253ca591662b3aac849da12d0...,2/10/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",NaN,mir hat der service gut gefallen.
1,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",NaN,great
2,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",NaN,convenient to shopping and dining options. go...
3,3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...,2/8/23,HOTEL,"{""overall"": 3.0, ""roomcleanliness"": 3.0, ""serv...",NaN,good regular clean place
4,823fb2499b4e37d99acb65e7198e75965d6496fd1c579f...,2/9/23,HOTEL,"{""overall"": 4.0, ""roomcleanliness"": 4.0, ""serv...",NaN,location was good


In [35]:
vocab_df = build_unique_amenity_vocab(description_df)
mapping_df = bootstrap_amenity_mapping(vocab_df)

# optional one-time model cleanup pass
mapping_df = canonicalize_amenity_vocab_with_model(
    client,
    vocab_df,
    model="gpt-5.4-mini",
    batch_size=150,
    use_rule_bootstrap=True,
)

display(mapping_df.head(30))

,raw_amenity_value,action,canonical_name,canonical_category,canonical_description,reason,normalized_raw_amenity_value
0,elevator,keep,elevator,accessibility,Elevator or lift access within the property.,core accessibility amenity,elevator
1,available in all rooms: free wifi,merge,free wifi,wifi,Free wireless internet available in guest rooms.,normalize internet wording to wifi concept,available in all rooms free wifi
2,internet,merge,wifi,wifi,Internet access or wireless internet availabil...,broad internet concept; useful for review matc...,internet
3,available in some public areas: free wifi,merge,free wifi,wifi,Free wireless internet available in some publi...,normalize internet wording to wifi concept,available in some public areas free wifi
4,english,drop,,,,language label does not describe an amenity co...,english
5,ac,merge,air conditioning,comfort,Guestroom or property air conditioning.,common abbreviation for air conditioning,ac
6,daily housekeeping,merge,housekeeping,service,"Housekeeping or room cleaning service, includi...",specific frequency should map to general house...,daily housekeeping
7,housekeeping,keep,housekeeping,service,Housekeeping or room cleaning service.,useful broad service amenity,housekeeping
8,if you have requests for specific accessibilit...,drop,,,,"policy/instruction text, not an amenity",if you have requests for specific accessibilit...
9,24-hour front desk,merge,24 hour front desk,service,Front desk staffed around the clock.,standardized time formatting,24 hour front desk


In [36]:
amenity_catalog_df = build_amenity_catalog(description_df)

print("Hotels:", amenity_catalog_df["eg_property_id"].nunique())
print("Amenity rows:", len(amenity_catalog_df))
display(amenity_catalog_df.head(20))

Hotels: 13
Amenity rows: 694


,amenity_id,eg_property_id,source_column,source_category,amenity_category,raw_amenity_value,amenity_name,mapping_action,amenity_embedding_text
0,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,ac,ac,keep,Hotel amenity mention candidate. Canonical cat...
1,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,bar,bar,keep,Hotel amenity mention candidate. Canonical cat...
2,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,breakfast_included,breakfast included,keep,Hotel amenity mention candidate. Canonical cat...
3,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,frontdesk_24_hour,frontdesk 24 hour,keep,Hotel amenity mention candidate. Canonical cat...
4,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,housekeeping,housekeeping,keep,Hotel amenity mention candidate. Canonical cat...
5,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,internet,internet,keep,Hotel amenity mention candidate. Canonical cat...
6,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,no_smoking,no smoking,keep,Hotel amenity mention candidate. Canonical cat...
7,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,outdoor_space,outdoor space,keep,Hotel amenity mention candidate. Canonical cat...
8,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,restaurant,restaurant,keep,Hotel amenity mention candidate. Canonical cat...
9,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,110f01b8ae518a0ee41047bce5c22572988a435e10ead7...,popular_amenities_list,popular amenities,popular amenities,tv,tv,keep,Hotel amenity mention candidate. Canonical cat...


In [37]:
chunk_df = build_review_chunk_table(reviews_df.head(10), max_chars=320)

print("Chunk rows:", len(chunk_df))
display(chunk_df.head(20))

Chunk rows: 10


,review_id,eg_property_id,acquisition_date,lob,rating,chunk_id,chunk_text
0,0,9a0043fd4258a1286db1e253ca591662b3aac849da12d0...,2/10/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",0,mir hat der service gut gefallen.
1,1,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",0,great
2,2,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",0,convenient to shopping and dining options. goo...
3,3,3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...,2/8/23,HOTEL,"{""overall"": 3.0, ""roomcleanliness"": 3.0, ""serv...",0,good regular clean place
4,4,823fb2499b4e37d99acb65e7198e75965d6496fd1c579f...,2/9/23,HOTEL,"{""overall"": 4.0, ""roomcleanliness"": 4.0, ""serv...",0,location was good
5,5,fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...,2/13/23,HOTEL,"{""overall"": 4.0, ""roomcleanliness"": 5.0, ""serv...",0,"we weren’t allowed to check in early, but that..."
6,7,fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...,2/13/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 0.0, ""serv...",0,very clean and great location
7,8,3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...,2/13/23,HOTEL,"{""overall"": 2.0, ""roomcleanliness"": 1.0, ""serv...",0,this property is filthy!! the walls of the roo...
8,8,3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...,2/13/23,HOTEL,"{""overall"": 2.0, ""roomcleanliness"": 1.0, ""serv...",1,"location is not good, but it was necessary for..."
9,9,3216b1b7885bffdb336265a8de7322ba0cd477cfb3d4f9...,2/26/23,HOTEL,"{""overall"": 3.0, ""roomcleanliness"": 3.0, ""serv...",0,it couldve been worse. we had to change rooms ...


## Retrieval-only smoke test

This lets you check whether the top candidates make sense **before** paying for verifier calls.


In [38]:
# Build hotel amenity index once.
amenity_embeddings = embed_texts(
    client,
    amenity_catalog_df["amenity_embedding_text"].tolist(),
    model="text-embedding-3-small",
)
hotel_index = build_hotel_amenity_index(amenity_catalog_df, amenity_embeddings)

# Pick one review to inspect manually.
sample_review = reviews_df.iloc[[1]].copy()
sample_chunks = build_review_chunk_table(sample_review, max_chars=320)
sample_chunk_embeddings = embed_texts(
    client,
    sample_chunks["chunk_text"].tolist(),
    model="text-embedding-3-small",
)

retrieved = retrieve_top_amenity_candidates(
    hotel_index=hotel_index,
    property_id=str(sample_review["eg_property_id"].iloc[0]),
    chunk_texts=sample_chunks["chunk_text"].tolist(),
    chunk_embeddings=sample_chunk_embeddings,
    top_k=5,
    min_similarity=0.4,
)

display(sample_review)
display(sample_chunks)
if retrieved.empty:
    display(retrieved)
else:
    display(retrieved.sort_values("retrieval_score", ascending=False))

,eg_property_id,acquisition_date,lob,rating,review_title,review_text
1,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",NaN,great


,review_id,eg_property_id,acquisition_date,lob,rating,chunk_id,chunk_text
0,0,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",0,great


,chunk_id,chunk_text,amenity_id,amenity_name,amenity_category,raw_amenity_value,retrieval_score


## Add the verifier

Default is `gpt-5.4-nano` to keep costs down.  
If you want more quality, switch to `gpt-5.4-mini`.


In [39]:
top_candidates = (
    retrieved
    if retrieved.empty or "retrieval_score" not in retrieved.columns
    else retrieved.sort_values("retrieval_score", ascending=False).head(8)
)

verified = verify_candidates_dataframe(
    client,
    top_candidates,
    model="gpt-5.4-nano",
)

display(verified)

""


## Run a small batch end-to-end

Start small. Tune your thresholds and prompts on a subset before scaling up.


In [47]:
sample_reviews = reviews_df.head(500).copy()

amenity_catalog_df, mentions_df, summary_df = process_reviews(
    client=client,
    description_df=description_df,
    reviews_df=sample_reviews,
    amenity_mapping_df=mapping_df,
    embedding_model="text-embedding-3-small",
    verifier_model="gpt-5.4-nano",
    top_k=3,
    min_similarity=0.4,
    verify=True,
    max_verifications_per_review=8,
)

display(mentions_df.head(30))
display(summary_df.head(30))

,chunk_id,chunk_text,amenity_id,amenity_name,amenity_category,raw_amenity_value,retrieval_score,review_id,eg_property_id,acquisition_date,lob,rating,mentioned,explicitness,sentiment,evidence,rationale_short
0,0,convenient to shopping and dining options. goo...,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,conference space,business services,conference space (1122 square feet),0.405563,2,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/14/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",False,not_mentioned,unknown,good business hotel.,The chunk mentions being a business hotel and ...
1,0,easy access from 75 clean breakfast included r...,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,free continental breakfast,dining,free continental breakfast available 6:00 am–9...,0.428353,10,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,3/1/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 0.0, ""serv...",True,implicit,positive,clean breakfast included,The chunk explicitly indicates breakfast is in...
2,0,easy access from 75 clean breakfast included r...,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,breakfast,dining,breakfast_included,0.408313,10,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,3/1/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 0.0, ""serv...",True,explicit,positive,clean breakfast included,The chunk explicitly says breakfast was included.
3,0,"reall clean, nice, and safe hotel. ample parki...",ff26cdda236b233f7c481f0e896814075ac6bed335e162...,free parking,parking,free self parking on site,0.417003,20,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/23/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",False,not_mentioned,positive,ample parking,"The chunk mentions parking availability, but i..."
4,0,"reall clean, nice, and safe hotel. ample parki...",ff26cdda236b233f7c481f0e896814075ac6bed335e162...,free parking,parking,free_parking,0.408771,20,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/23/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",False,not_mentioned,positive,ample parking,"The chunk mentions parking availability, but i..."
5,0,the entire hotel property provides lots of lar...,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,conference space,business services,conference space (40000 square feet),0.446768,56,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/28/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",True,implicit,positive,provides lots of large spaces for hanging out ...,The chunk mentions large on-property spaces su...
6,0,the entire hotel property provides lots of lar...,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,ballroom,event,ballroom,0.443991,56,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,2/28/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 4.0, ""serv...",True,implicit,positive,the entire hotel property provides lots of lar...,The chunk mentions large on-site spaces for en...
7,0,super convenient and stylish hotel. absolutely...,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,24 hour front desk,service,frontdesk_24_hour,0.402635,59,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/28/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",False,not_mentioned,positive,absolutely loved the staff. so welcoming and v...,The chunk praises staff hospitality but does n...
8,1,"breakfast was not included, but enjoyed a beau...",ff26cdda236b233f7c481f0e896814075ac6bed335e162...,cooked-to-order breakfast available for a fee,dining,cooked-to-order breakfast available for a fee ...,0.436183,69,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,2/28/23,HOTEL,"{""overall"": 5.0, ""roomcleanliness"": 5.0, ""serv...",True,implicit,positive,enjoyed a beautifully presented breakfast at t...,The guest refers to having breakfast at the ho...
9,1,"breakfast was not included, but enjoyed a beau...",ff26cdda236b233f7c481f0e896814075ac6bed335e162...,breakfast,dining,breakfast_available,0.420187,69,ff2

,review_id,amenity_id,amenity_name,amenity_category,eg_property_id,best_retrieval_score,mention_count,sample_evidence,sentiment
0,10,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,free continental breakfast,dining,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,0.428353,1,clean breakfast included,positive
1,10,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,breakfast,dining,7d027ef72c02eaa17af3c993fd5dba50d17b41a6280389...,0.408313,1,clean breakfast included,positive
2,56,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,conference space,business services,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,0.446768,1,provides lots of large spaces for hanging out ...,positive
3,56,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,ballroom,event,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,0.443991,1,the entire hotel property provides lots of lar...,positive
4,69,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,cooked-to-order breakfast available for a fee,dining,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,0.436183,1,enjoyed a beautifully presented breakfast at t...,positive
5,69,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,breakfast,dining,ff26cdda236b233f7c481f0e896814075ac6bed335e162...,0.420187,1,"breakfast was not included, but enjoyed a beau...",mixed
6,74,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,free area shuttle,transportation,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,0.451478,1,the shuttle was very convenient,positive
7,74,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,free area shuttle,transport,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,0.437545,1,the shuttle was very convenient,positive
8,79,fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...,covered self parking,parking,fa014137b3ea9af6a90c0a86a1d099e46f7e56d6eb33db...,0.415399,1,the parking is in a garage underneath the hotel.,neutral
9,129,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,hot tub,pool,db38b19b897dbece3e34919c662b3fd66d23b615395d11...,0.424819,1,there is an outdoor hot tub which was pleasant...,positive


In [48]:
OUTPUT_DIR = "outputs"

amenity_catalog_df.to_csv("outputs/amenity_catalog.csv", index=False)
mentions_df.to_csv("outputs/review_chunk_mentions.csv", index=False)
summary_df.to_csv("outputs/review_amenity_summary.csv", index=False)

print("Wrote files to", OUTPUT_DIR)

Wrote files to outputs


## Suggested next steps

- Tune `min_similarity` by language and by amenity family
- Add a human-labeled eval set for precision / recall
- Cache amenity embeddings per hotel so you only re-embed when descriptions change
- Batch verifier calls offline for backfills; keep online spend bounded with `max_verifications_per_review`
- Consider a 3-band policy:
  - very high similarity → maybe skip verifier for exact matches
  - medium similarity → send to verifier
  - low similarity → reject
